In [ ]:
"""
Author: Aquiles Elbaum

Description:
    Retrieval-augmented generation pipeline for sensor recommendation using
    a FAISS vector store and Gemma, with attention matrix extraction.
"""

# =========================
# 1) Auth + Drive Mount
# =========================
from google.colab import drive
from google.colab import userdata
from huggingface_hub import login

drive.mount("/content/drive")

login(token=userdata.get("HUGGINGFACE_TOKEN"))

In [ ]:
# =========================
# 2) Imports
# =========================
import os
import json
import torch
import pickle
import pandas as pd
from datetime import datetime
from tqdm import tqdm

import faiss
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

In [ ]:
# =========================
# 3) Paths
# =========================
PROJECT_DIR = "/content/drive/MyDrive/Capstone/RAG_Run"

FAISS_DIR = os.path.join(PROJECT_DIR, "Vectorstore_AllPdfs")  # contains index.faiss + index.pkl
CSV_PATH  = os.path.join(PROJECT_DIR, "mines.csv") # Dataset with context and target

# Output root
OUT_ROOT = os.path.join(PROJECT_DIR, "RAG_Output")
os.makedirs(OUT_ROOT, exist_ok=True)

In [ ]:
# =========================
# 4) GPU / Device
# =========================
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU (will be slow).")

# We'll rely on model.device later; keep a fallback
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# =========================
# 5) Load FAISS vectorstore
# =========================
emb_device = device

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": emb_device}
)

# Load metadata tuple (docstore, index_to_docstore_id)
with open(os.path.join(FAISS_DIR, "index.pkl"), "rb") as f:
    docstore, index_to_docstore_id = pickle.load(f)

# Load FAISS index
faiss_index = faiss.read_index(os.path.join(FAISS_DIR, "index.faiss"))

vectorstore = FAISS(
    embedding_function=embedding_model,
    index=faiss_index,
    docstore=docstore,
    index_to_docstore_id=index_to_docstore_id,
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 6})

def retrieve_context(context: str, target: str) -> str:
    query = f"{context} detect landmine {target}"
    docs = retriever.get_relevant_documents(query)
    return "\n".join([d.page_content[:500] for d in docs])  # hard truncate per-doc

In [ ]:
# =========================
# 6) Load Gemma-3 with attention enabled
# =========================
model_id = "google/gemma-3-4b-it"

cfg = AutoConfig.from_pretrained(model_id)
cfg.output_attentions = True

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    config=cfg,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    attn_implementation="eager",   # F attention extraction
    device_map="auto"
)

# Attentions + cache can break / return NaNs
model.config.use_cache = False
try:
    model.generation_config.cache_implementation = None
except Exception:
    pass

model.eval()

print("Model device:", model.device)

In [ ]:
# =========================
# 7) Load data + output dirs
# =========================
df = pd.read_csv(CSV_PATH)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
base_dir = os.path.join(OUT_ROOT, timestamp)
attn_dir = os.path.join(base_dir, "attn")
os.makedirs(attn_dir, exist_ok=True)

out_csv = os.path.join(base_dir, "sensor_recommendations.csv")
checkpoint_path = os.path.join(base_dir, "resume_checkpoint.txt")

# Resume
last_done = -1
if os.path.exists(checkpoint_path):
    with open(checkpoint_path, "r") as f:
        last_done = int(f.read().strip() or -1)

print(f"Resuming from row {last_done + 1}")
print("Output CSV:", out_csv)

# If resuming, append mode; else write header once
write_header = not os.path.exists(out_csv)

In [ ]:
# =========================
# 8) Inference
# =========================
@torch.inference_mode()
def run_inference(prompt: str, idx: int):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=200,
        temperature=0.0,
        do_sample=False,
        return_dict_in_generate=True,
        output_attentions=True,
    )

    # Decode only the newly generated tokens
    input_len = inputs["input_ids"].shape[-1]
    new_tokens = outputs.sequences[0][input_len:]
    gen_text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # Attentions structure: list over generation steps,
    # each step: tuple of layers,
    # each layer: (batch, heads, q_len, k_len)
    if outputs.attentions is None:
        raise RuntimeError("No attentions returned. Ensure attn_implementation='eager' and use_cache=False.")

    final_step_attn = outputs.attentions[-1]   # last generated token step
    last_layer_attn = final_step_attn[-1]      # last layer tensor [B, H, Q, K]
    attn_mean = last_layer_attn.mean(dim=1).squeeze().float().cpu().tolist()

    # NaN guard
    if any((x != x) for x in (attn_mean if isinstance(attn_mean, list) else [attn_mean])):
        raise RuntimeError("Attention contains NaNs. Double-check eager attention + cache disabled.")

    attn_path = os.path.join(attn_dir, f"attn_{idx:06d}.json")
    with open(attn_path, "w") as f:
        json.dump(attn_mean, f)

    return gen_text, attn_path

In [ ]:
# =========================
# 9) Main loop (stream to CSV)
# =========================
for i, row in tqdm(df.iterrows(), total=len(df), desc="Processing"):
    if i <= last_done:
        continue

    c = str(row["Context"])
    t = str(row["Target"])

    retrieved = retrieve_context(c, t)

    prompt = (
        f"Evidence:\n{retrieved}\n\n"
        f"Recommend the best sensor.\n"
        f"Context: {c}\n"
        f"Target: {t}\n"
        f"You are a sensor expert.\n"
    )

    response, attn_file = run_inference(prompt, i)

    out_row = pd.DataFrame([{
        "context": c,
        "target": t,
        "response": response,
        "attention": attn_file
    }])

    out_row.to_csv(out_csv, mode="a", header=write_header, index=False)
    write_header = False

    with open(checkpoint_path, "w") as f:
        f.write(str(i))

print("Output saved:", out_csv)
print("Attentions saved in:", attn_dir)